## How to use this notebook

Run notebooks 01 and 02 first, then `Run All` here. It groups same-name recipe rows into versions,
prints the comparison for Chicken Inasal, and shows which row "fastest" selects. For a plain
terminal version of the same flow, run `python examples/demo_answer.py` from the repository root.

# 05 · Same-name recipe versions

Rows that share an exact `recipe_name` but contain different ingredients and instructions are
**versions of the same dish**, not duplicates to delete. This notebook uses `src/dish_groups.py`
to group them, merge near-identical rows, and produce a deterministic comparison the user can
choose from before the RAG step.

Run notebooks 01 and 02 first so `data/processed/cleaned_recipes.parquet` has
`canonical_ingredients`. If that file is missing, this notebook falls back to the older snapshot
under `notebooks/data/processed/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

primary = ROOT / "data/processed/cleaned_recipes.parquet"
fallback = ROOT / "notebooks/data/processed/cleaned_recipes.parquet"
path = primary if primary.exists() else fallback
df = pd.read_parquet(path)
print(f"Loaded {len(df)} recipes from {path.relative_to(ROOT)}")
print("canonical_ingredients present:", "canonical_ingredients" in df.columns)

In [ ]:
from dish_groups import (build_groups, format_version_comparison, select_version,
                         version_facts, version_positions)

groups = build_groups(df)
multi = [g for g in groups if len(g.rows) > 1]
merged = sum(len(g.merged_rows) for g in groups)
print(f"{len(groups)} dish groups | {len(multi)} with multiple versions | {merged} near-identical rows merged")

for g in sorted(multi, key=lambda g: -len(g.rows))[:8]:
    print(f"  {len(g.rows)} versions: {g.display_name}")

In [ ]:
# Reproduction of the three-version comparison, using the real Chicken Inasal rows.
target = [g for g in groups if g.name_key == "chicken inasal"]
assert target, "No 'chicken inasal' rows in this parquet snapshot"
group = target[0]

pantry = ["chicken", "vinegar", "garlic", "onion"]  # example pantry
eligible = version_positions(group, df, pantry)     # exactly the versions shown below
print(format_version_comparison(group, df, pantry))

In [ ]:
# Selection stays in code; only the chosen row reaches the LLM context.
for label, choice in [("fastest", "fastest"), ("fewest missing", "fewest missing")]:
    pos = select_version(group, df, pantry, choice, eligible=eligible)
    row = df.iloc[pos]
    facts = version_facts(row, pantry)
    print(f"{label}: row {pos} | {row['recipe_name']} | {facts['display_time']} | "
          f"missing: {', '.join(facts['missing']) or 'none'}")
    print("  ingredients:", str(row["ingredients"])[:180])

In [ ]:
chosen = df.iloc[select_version(group, df, pantry, "fastest", eligible=eligible)]
context_block = (
    f"[Recipe] {chosen['recipe_name']}\n"
    f"Ingredients: {chosen['ingredients']}\n"
    f"Instructions: {str(chosen['instructions'])[:1200]}"
)
print(context_block[:900])

`total_time` is shown as reported in the dataset; `active` minutes come from `cook_time_min`
and are what "I only have 15 minutes to cook" should be matched against. Datasets can disagree
between the two fields, so both are displayed.

## Integrating with notebook 04

1. Call `select_version(...)` after the user answers the comparison prompt.
2. Pass only that row to `format_context()` in `04_rag_pipeline.ipynb`.
3. Change `TOP_K` to count **dish groups** (via `rank_groups`) rather than rows, so two versions
   of the same name never occupy two recommendation slots.
4. At evaluation time, grade all versions of a name as equally relevant and report Recall@k
   and nDCG@k over groups.